# MediOps — The Same Curriculum, Rebuilt on Microsoft Agent Framework

Same fictional-company concept, same drug-batch journey, same 11 chapters
— this time using **Microsoft Agent Framework** (the successor to Semantic
Kernel + AutoGen). Company name: **MediCore Inc.**

## Why the code looks different again

Microsoft Agent Framework's building blocks map onto our recurring
patterns quite directly, but with its own vocabulary:

- **Single tool-using agents** are a `ChatClient.as_agent(tools=[...])` call
  — conceptually identical to ADK's `Agent(tools=[...])`, different names.
- **Supervisor/coordinator routing (Ch.4, Ch.9)** uses a purpose-built
  **`HandoffBuilder`** — a named orchestration pattern for exactly "one
  agent hands off to the right specialist," which neither LangGraph nor
  ADK has as a dedicated builder (LangGraph: manual routing model +
  conditional edges; ADK: implicit `sub_agents` delegation).
- **Fixed pipelines (Ch.5)** use a purpose-built **`SequentialBuilder`** —
  again, a named pattern rather than something you assemble from
  general-purpose graph primitives.
- **Reflection loops (Ch.8, Ch.10)** use the **Functional Workflow API**
  (`@workflow` / `@step`) — plain Python `while` loops and `if`/`else`
  calling `.run()` on each agent directly. This is a deliberate framework
  choice: Microsoft's own docs say to reach for graph concepts only when
  plain Python control flow won't do, so a draft-review-revise loop is
  written as... a loop.
- **Example 1 stays non-agentic on purpose** — plain Python, no framework
  — same teaching point as the other two notebooks.

## The Drug Manufacturing Journey — where each agent fits

```
RAW MATERIAL INTAKE  ->  FORMULATION  ->  QUALITY CONTROL
Ch.1 Batch Tracker        Ch.1 Batch        Ch.3 Batch Release Agent
Ch.7 Inventory Agent      Tracker           (ChatAgent + tools)
                                                  |
                                      +-----------+-----------+
                                    PASS                    FAIL
                                      |                       |
                          REGULATORY REVIEW           DEVIATION HANDLING
                          Ch.4 HandoffBuilder ->        Ch.8 Functional
                          regulatory_agent               Workflow loop
                                      |                  (draft -> review)
                          BATCH DOCUMENTATION                  |
                          Ch.4 HandoffBuilder ->         back to FORMULATION
                          documentation_agent
                                      |
                          HUMAN PHARMACIST SIGN-OFF    <- agents inform,
                                      |                    human decides
                          MARKET-FACING DRUG INFO
                          Ch.6 Drug Info & Pricing Agent
                                      |
                          POST-MARKET SURVEILLANCE
                          Ch.10 Adverse Event Screener (Functional Workflow)
                                      |
                          ONGOING COMPLIANCE
                          Ch.11 Regulatory Guidance Summarizer

R&D (Ch.5, SequentialBuilder) feeds new compounds INTO raw material intake.
SOP Assistant (Ch.2) and Plant Operations Router (Ch.9, HandoffBuilder)
run alongside every stage above.
```

## How to run this notebook

1. Run **Setup** to install `agent-framework`.
2. Run **Config** and choose a chat client provider.
   - `openai` needs `OPENAI_API_KEY`.
   - `ollama` runs free and local (Agent Framework ships an Ollama chat
     client) — install [Ollama](https://ollama.com/download), then
     `ollama pull llama3.2`, before running.
3. Run cells top to bottom — later chapters reuse agents built earlier.

**Note on API surface:** Microsoft Agent Framework is a fast-moving,
recently-GA'd library (successor to Semantic Kernel + AutoGen). If an
import or method name below has shifted slightly in your installed
version, check `https://learn.microsoft.com/en-us/agent-framework/` for
the current signature — the concepts and structure will still map
directly even if a method name has moved.


## Setup

In [ ]:
%pip install -q agent-framework pandas pydantic python-dotenv
print("Dependencies installed.")

## Config — choose your chat client provider

`get_chat_client()` is used everywhere below, so this is the only place
you need to change providers.

In [ ]:
import os

CLIENT_PROVIDER = "ollama"   # "ollama" or "openai"
OLLAMA_MODEL = "llama3.2"
OPENAI_MODEL = "gpt-4o-mini"

# If using openai: set your key (or export OPENAI_API_KEY before starting Jupyter)
# os.environ["OPENAI_API_KEY"] = "your-key-here"

def get_chat_client():
    """Returns the chat client configured by CLIENT_PROVIDER above."""
    if CLIENT_PROVIDER == "ollama":
        from agent_framework.ollama import OllamaChatClient
        return OllamaChatClient(model_id=OLLAMA_MODEL)
    else:
        from agent_framework.openai import OpenAIChatClient
        return OpenAIChatClient(model_id=OPENAI_MODEL)

print(f"Provider: {CLIENT_PROVIDER}")

---
# Example 1 — Batch Stage Tracker (not an agent)

**Use case:** Before MediOps can make any decision about a batch, it needs
a record of what's happened to that batch so far. Deliberately plain
Python — no LLM, no framework — because there's no reasoning or decision
to make. Same non-agentic baseline as the other two notebooks.

In [ ]:
def log_material_receipt(batch_state: dict) -> dict:
    """Record that raw materials for this batch have arrived and been logged."""
    batch_state["stage_log"] += " -> Raw materials received"
    print(f"[MATERIAL RECEIPT] {batch_state['stage_log']}")
    return batch_state

def log_formulation_complete(batch_state: dict) -> dict:
    """Record that the batch has been formulated per the master batch record."""
    batch_state["stage_log"] += " -> Formulation complete"
    print(f"[FORMULATION] {batch_state['stage_log']}")
    return batch_state

def log_qc_check(batch_state: dict) -> dict:
    """Record that the batch has passed through the quality control checkpoint."""
    batch_state["stage_log"] += " -> QC checkpoint passed"
    print(f"[QUALITY CONTROL] {batch_state['stage_log']}")
    return batch_state

batch_state = {"batch_id": "MC-2026-0091", "stage_log": "Batch opened"}
batch_state = log_material_receipt(batch_state)
batch_state = log_formulation_complete(batch_state)
batch_state = log_qc_check(batch_state)
print()
print(batch_state)

---
# Example 2 — SOP Assistant

**Use case:** Plant operators need to check Standard Operating Procedures
without pulling a supervisor off the floor. First real Agent Framework
agent: `client.as_agent(...)` with no tools, just a persona. Conversation
memory is kept as a running message list, appended to and replayed each
turn — the same mechanic you saw explicitly in the LangGraph notebook,
just without a graph wrapping it.

In [ ]:
sop_assistant_agent = get_chat_client().as_agent(
    name="SOPAssistant",
    instructions="""You are the MediCore SOP Assistant. You help plant
    operators understand standard operating procedures for batch handling,
    cleanroom protocol, and GMP documentation requirements. Be precise, and
    say so if a procedure isn't in your knowledge rather than guessing.""",
)

conversation_history = []

async def ask_sop_assistant(question: str):
    global conversation_history
    conversation_history.append(question)
    result = await sop_assistant_agent.run(question)
    conversation_history.append(str(result))
    print(f"Operator: {question}")
    print(f"SOP Assistant: {result}\n")

await ask_sop_assistant("What's the gowning procedure before entering a Class B cleanroom?")
await ask_sop_assistant("And how often does that gown need to be changed?")

---
# Example 3 — Batch Release Agent

**Use case:** A pharmacist asks "can this batch be released?" This
requires *acting* — the agent must pull lab purity data and check dosage
math before responding. Same as ADK: `as_agent(tools=[...])` runs the
reason -> call tool -> observe -> reason loop internally, no manual graph
wiring needed.

In [ ]:
def check_batch_purity(batch_id: str) -> str:
    """Look up lab purity test results for a given batch ID."""
    lab_results = {
        "MC-2026-0091": "Purity: 99.6% | Impurity profile: within spec | Status: PASS",
        "MC-2026-0044": "Purity: 96.1% | Impurity profile: exceeds threshold | Status: FAIL",
    }
    return lab_results.get(batch_id, f"No lab record found for batch {batch_id}")

def calculate_dosage_variance(target_mg: float, measured_mg: float) -> str:
    """Calculate the % variance between target and measured dosage per unit."""
    variance_pct = abs(measured_mg - target_mg) / target_mg * 100
    verdict = "within USP tolerance" if variance_pct <= 5 else "OUT OF TOLERANCE"
    return f"Variance: {variance_pct:.2f}% - {verdict}"

def lookup_drug_monograph(drug_name: str) -> str:
    """Retrieve the regulatory monograph summary for a drug (dosage form, storage, indications)."""
    monographs = {
        "metformin": "Oral biguanide, 500-1000mg tablets, store below 25C, indicated for T2DM.",
    }
    return monographs.get(drug_name.lower(), f"No monograph found for {drug_name}")

batch_release_agent = get_chat_client().as_agent(
    name="BatchReleaseAgent",
    instructions="""You are the MediCore Batch Release Agent. Before
    answering any release question, use your tools to check actual lab
    data - never guess purity or dosage figures.""",
    tools=[check_batch_purity, calculate_dosage_variance, lookup_drug_monograph],
)

result = await batch_release_agent.run(
    "Is batch MC-2026-0091 cleared for release? Check purity and confirm "
    "dosage variance for target 500mg, measured 512mg."
)
print(result)

---
# Example 4 — Manufacturing Supervisor (HandoffBuilder)

**Use case:** A single agent juggling purity checks, regulatory lookups,
and document drafting gets confused about which tool applies. MediOps
splits into specialists and uses Agent Framework's purpose-built
**`HandoffBuilder`** — a named orchestration pattern for exactly this
"triage agent hands off to the right specialist" shape.

In [ ]:
from agent_framework_orchestrations import HandoffBuilder

quality_control_agent = get_chat_client().as_agent(
    name="QualityControlAgent",
    instructions="""You verify batch purity and dosage variance against
    spec. Always check the actual lab data via tools before giving a
    verdict.""",
    tools=[check_batch_purity, calculate_dosage_variance],
)

regulatory_agent = get_chat_client().as_agent(
    name="RegulatoryAgent",
    instructions="""You answer questions about drug monographs and
    compliance classification. Always consult the monograph tool before
    answering.""",
    tools=[lookup_drug_monograph],
)

documentation_agent = get_chat_client().as_agent(
    name="DocumentationAgent",
    instructions="""You draft clear, GMP-compliant batch records and
    summaries from information already gathered in the conversation.""",
)

manufacturing_supervisor_agent = get_chat_client().as_agent(
    name="ManufacturingSupervisor",
    instructions="""You receive a pharmacist's question and hand it off to
    the right specialist: QualityControlAgent for purity/dosage,
    RegulatoryAgent for monographs/compliance, DocumentationAgent for
    drafting records.""",
)

manufacturing_workflow = (
    HandoffBuilder(participants=[manufacturing_supervisor_agent, quality_control_agent,
                                  regulatory_agent, documentation_agent])
    .with_start_agent(manufacturing_supervisor_agent)
    .add_handoff(manufacturing_supervisor_agent,
                 [quality_control_agent, regulatory_agent, documentation_agent])
    .build()
)

result = await manufacturing_workflow.run(
    "Is batch MC-2026-0091 within dosage tolerance for target 500mg measured 512mg, "
    "and what's the monograph classification for metformin?"
)
print(result)

---
# Example 5 — Formulation R&D Pipeline (SequentialBuilder)

**Use case:** Before MediCore reformulates a drug, R&D needs a structured
literature review — plan sub-questions, search literature for each,
synthesize gaps, write a report. `SequentialBuilder` chains agents in a
fixed pipeline; by default each agent sees the full prior conversation, so
context threads through automatically — no explicit state schema needed.

In [ ]:
from agent_framework_orchestrations import SequentialBuilder

def search_pharma_literature(query: str) -> str:
    """Search pharmaceutical literature and patents for a formulation topic."""
    mock_results = {
        "bioavailability": "3 relevant papers on solubility enhancement via "
                            "nanocrystal formulation and lipid-based delivery.",
        "solubility": "2 papers on cyclodextrin complexation improving aqueous solubility.",
        "stability": "1 paper on polymorph screening to improve thermal stability.",
    }
    for keyword, result in mock_results.items():
        if keyword in query.lower():
            return result
    return f"Limited results for '{query}'."

formulation_planner_agent = get_chat_client().as_agent(
    name="FormulationPlanner",
    instructions="""Break the research question into 3-4 short, specific
    subtasks, one per line, numbered.""",
)

literature_searcher_agent = get_chat_client().as_agent(
    name="LiteratureSearcher",
    instructions="""For each subtask given to you, call
    search_pharma_literature and summarize the findings.""",
    tools=[search_pharma_literature],
)

gap_analyzer_agent = get_chat_client().as_agent(
    name="GapAnalyzer",
    instructions="""Synthesize the literature findings so far and identify
    open formulation gaps.""",
)

rd_report_writer_agent = get_chat_client().as_agent(
    name="RDReportWriter",
    instructions="""Write a short formulation R&D report with sections:
    Overview, Key Findings, Formulation Gaps, Recommended Next Steps.""",
)

formulation_rd_pipeline = SequentialBuilder(
    participants=[formulation_planner_agent, literature_searcher_agent,
                  gap_analyzer_agent, rd_report_writer_agent]
).build()

result = await formulation_rd_pipeline.run(
    "How can we improve the bioavailability of Compound X?")
print(result)

---
# Example 6 — Drug Info & Pricing Assistant

**Use case:** Patients and pharmacists ask two kinds of questions once a
drug is released: "what does it treat" and "how much does it cost." A
single tool-using agent handles both; conversation history threading
(kept as a running list, same mechanic as Example 2) lets a follow-up
like "how much does it cost?" resolve to whichever drug was named earlier.

In [ ]:
import pandas as pd

drug_price_table = pd.DataFrame([
    {"drug_name": "Metformin 500mg", "price_usd": 4.50},
    {"drug_name": "Amoxicillin 250mg", "price_usd": 6.20},
    {"drug_name": "Atorvastatin 20mg", "price_usd": 9.85},
])

drug_monograph_texts = {
    "metformin": "Metformin: oral biguanide for type 2 diabetes. Typical dose "
                 "500-1000mg twice daily with meals. Store below 25C. Common "
                 "side effect: GI upset.",
    "amoxicillin": "Amoxicillin: penicillin-class antibiotic. Typical dose "
                    "250-500mg every 8 hours. Avoid in penicillin allergy.",
    "atorvastatin": "Atorvastatin: statin for cholesterol management. Typical "
                     "dose 10-20mg once daily, evening administration preferred.",
}

def get_drug_price(drug_name: str) -> str:
    """Look up the retail price of a drug by name (substring match)."""
    matches = drug_price_table[
        drug_price_table["drug_name"].str.contains(drug_name, case=False, na=False)
    ]
    if matches.empty:
        return f"No pricing found for '{drug_name}'."
    row = matches.iloc[0]
    return f"{row['drug_name']}: ${row['price_usd']:.2f}"

def get_drug_info(drug_name: str) -> str:
    """Retrieve indications, dosage, and warnings for a drug from its monograph."""
    for key, text in drug_monograph_texts.items():
        if key in drug_name.lower():
            return text
    return f"No monograph found for '{drug_name}'."

drug_info_pricing_agent = get_chat_client().as_agent(
    name="DrugInfoPricingAgent",
    instructions="""You answer questions about what a drug treats, its
    dosage, warnings, and its price. Use get_drug_info for monograph
    questions and get_drug_price for cost questions. If the patient refers
    to "it" or "that drug", infer which drug from earlier in the
    conversation you're given.""",
    tools=[get_drug_price, get_drug_info],
)

patient_conversation = []

async def ask_drug_info_pricing(question: str):
    global patient_conversation
    patient_conversation.append(question)
    result = await drug_info_pricing_agent.run(patient_conversation)
    patient_conversation.append(str(result))
    return result

r1 = await ask_drug_info_pricing("What is Metformin used for and what's the typical dose?")
print("Patient: What is Metformin used for and what's the typical dose?")
print("Assistant:", r1, "\n")

r2 = await ask_drug_info_pricing("How much does it cost?")
print("Patient: How much does it cost?")
print("Assistant:", r2)

---
# Example 7 — Raw Material Inventory Agent

**Use case:** Before formulation can start, the system needs to check and
decrement raw material stock. Same as ADK, no manual graph needed — read
and write tools are both just Python functions on one `as_agent(...)`
call.

In [ ]:
raw_material_stock = pd.DataFrame([
    {"material_id": "RM-101", "quantity_kg": 250.0},
    {"material_id": "RM-204", "quantity_kg": 80.0},
])

def get_material_stock(material_id: str) -> str:
    """Query current stock level (kg) for a raw material by ID."""
    row = raw_material_stock[raw_material_stock["material_id"] == material_id]
    if row.empty:
        return f"No stock record for {material_id}"
    return f"{material_id}: {row.iloc[0]['quantity_kg']} kg available"

def consume_material(material_id: str, quantity_kg: float) -> str:
    """Deduct quantity_kg from a raw material's stock when a batch enters formulation."""
    idx = raw_material_stock.index[raw_material_stock["material_id"] == material_id]
    if len(idx) == 0:
        return f"No stock record for {material_id}"
    raw_material_stock.loc[idx, "quantity_kg"] -= quantity_kg
    remaining = raw_material_stock.loc[idx, "quantity_kg"].values[0]
    return f"Consumed {quantity_kg}kg of {material_id}. Remaining: {remaining}kg"

inventory_agent = get_chat_client().as_agent(
    name="InventoryAgent",
    instructions="""You manage raw material inventory. Use
    get_material_stock to check levels and consume_material to deduct
    stock when a batch enters formulation. Always confirm the resulting
    stock level after consuming.""",
    tools=[get_material_stock, consume_material],
)

result = await inventory_agent.run(
    "How much RM-101 do we have, and consume 30kg of it for the next batch?")
print(result)

---
# Example 8 — Batch Deviation Report Agent (Functional Workflow / plain loop)

**Use case:** When a batch fails QC, GMP requires a formal Deviation
Report with root cause and CAPA. This is where Agent Framework's design
philosophy shows most clearly: rather than a `LoopAgent` builder, its own
docs recommend plain Python control flow for exactly this shape. So the
draft -> review -> revise cycle is written as... a `while` loop calling
`.run()` on each agent directly.

In [ ]:
deviation_drafter_agent = get_chat_client().as_agent(
    name="DeviationDrafter",
    instructions="""Draft a GMP batch deviation report with sections:
    Incident Summary, Root Cause, Impact Assessment, CAPA (Corrective and
    Preventive Action). If review feedback is provided, revise the report
    to address it. Be specific and auditable.""",
)

deviation_reviewer_agent = get_chat_client().as_agent(
    name="DeviationReviewer",
    instructions="""Critique the deviation report draft you're given: is
    the root cause specific enough? Is the CAPA actionable? If it meets
    GMP documentation standards, respond with exactly 'APPROVED'.
    Otherwise, list concrete gaps for the drafter to fix.""",
)

async def run_deviation_report_loop(incident_notes: str, max_rounds: int = 3) -> str:
    """
    Plain-Python reflection loop: draft, review, and revise a deviation
    report up to max_rounds times, exiting early once the reviewer
    responds with APPROVED.
    """
    draft = await deviation_drafter_agent.run(incident_notes)
    for round_num in range(max_rounds):
        review = await deviation_reviewer_agent.run(str(draft))
        print(f"[Round {round_num + 1}] Reviewer: {str(review)[:100]}...")
        if "APPROVED" in str(review):
            break
        draft = await deviation_drafter_agent.run(
            f"Original draft:\n{draft}\n\nReviewer feedback:\n{review}\n\nRevise the report."
        )
    return str(draft)

final_report = await run_deviation_report_loop(
    "Batch MC-2026-0044 failed QC: purity 96.1%, impurity profile exceeded "
    "threshold. Root cause suspected: mixing time under-run by 12 minutes on "
    "Line 2. Draft the deviation report."
)
print("\nFINAL REPORT:\n", final_report)

---
# Example 9 — Plant Operations Router (HandoffBuilder, reusing earlier agents)

**Use case:** A plant operator's chat interface shouldn't require them to
know which specialist to talk to. MediOps composes the **Drug Info &
Pricing Agent** (Example 6) and the **Inventory Agent** (Example 7) as
`HandoffBuilder` participants — the exact agent objects built earlier, no
new copies, same composition story as the ADK version.

In [ ]:
plant_operations_router_agent = get_chat_client().as_agent(
    name="PlantOperationsRouter",
    instructions="""You receive plant operator questions and hand off to
    the right specialist: DrugInfoPricingAgent for drug info/pricing
    questions, InventoryAgent for raw material stock questions. For
    greetings or small talk, answer directly and briefly.""",
)

plant_operations_workflow = (
    HandoffBuilder(participants=[plant_operations_router_agent,
                                  drug_info_pricing_agent, inventory_agent])
    .with_start_agent(plant_operations_router_agent)
    .add_handoff(plant_operations_router_agent, [drug_info_pricing_agent, inventory_agent])
    .build()
)

result = await plant_operations_workflow.run("How much RM-204 do we have left?")
print(result)

---
# Example 10 — Adverse Event Literature Screener (batch reflection)

**Use case:** MediCore's pharmacovigilance team screens case reports for a
safety review, applying an inclusion criterion that's easy to misapply on
a single pass. Reuses the same plain-Python reflection loop shape from
Example 8, run once per report across a small in-memory batch.

In [ ]:
import re

INCLUSION_CRITERION = """
Exclude case reports lacking a causality assessment - EXCLUDING reports
sourced from recognized regulatory adverse-event databases (e.g., FAERS),
which are pre-vetted and should NOT be excluded on this basis.
"""

screener_agent = get_chat_client().as_agent(
    name="ScreenerAgent",
    instructions=f"""Criterion: {INCLUSION_CRITERION}
    If review feedback is provided, reconsider your decision. Respond with
    exactly: Decision: INCLUDE/EXCLUDE  Reason: <one sentence>""",
)

screening_reviewer_agent = get_chat_client().as_agent(
    name="ScreeningReviewer",
    instructions="""Check the screening decision you're given: did the
    screener correctly avoid excluding a FAERS-sourced report just for
    lacking a causality assessment? If correct, respond with exactly
    'APPROVED'. Otherwise explain the error for the screener to fix.""",
)

async def screen_one_case_report(source: str, narrative: str, max_rounds: int = 3) -> dict:
    """Runs the screen-review-revise loop for a single case report."""
    decision_text = await screener_agent.run(f"Source: {source}\n{narrative}")
    for _ in range(max_rounds):
        review = await screening_reviewer_agent.run(str(decision_text))
        if "APPROVED" in str(review):
            break
        decision_text = await screener_agent.run(
            f"Source: {source}\n{narrative}\n\nReviewer feedback:\n{review}\n\nReconsider."
        )
    decision = re.search(r"Decision:\s*(INCLUDE|EXCLUDE)", str(decision_text))
    reason = re.search(r"Reason:\s*(.+)", str(decision_text))
    return {
        "decision": decision.group(1) if decision else "UNKNOWN",
        "reason": reason.group(1) if reason else str(decision_text)[:200],
    }

sample_case_reports = pd.DataFrame([
    {"report_id": "AE-001", "source": "FAERS",
     "narrative": "Patient reported nausea after Metformin. No causality assessment on file."},
    {"report_id": "AE-002", "source": "Physician self-report",
     "narrative": "Patient reported dizziness. No causality assessment provided."},
])

async def screen_all_case_reports(case_reports: pd.DataFrame) -> pd.DataFrame:
    results = []
    for _, report in case_reports.iterrows():
        outcome = await screen_one_case_report(report["source"], report["narrative"])
        results.append({"report_id": report["report_id"], **outcome})
    return pd.DataFrame(results)

screening_results = await screen_all_case_reports(sample_case_reports)
screening_results

---
# Example 11 — Regulatory Guidance Summarizer

**Use case:** Regulatory affairs tracks dozens of FDA/EMA guidance
documents per year. Each needs a structured summary. A single agent call
— no orchestration needed, matching this chapter's honest "this is really
just one LLM call" shape in every notebook version so far.

In [ ]:
guidance_summarizer_agent = get_chat_client().as_agent(
    name="GuidanceSummarizer",
    instructions="""Summarize the regulatory guidance document you're
    given with exactly these sections: Scope, Key Requirements, Compliance
    Actions, Effective Date/Deadline.""",
)

sample_guidance_text = """
FDA Guidance for Industry: Process Validation for Solid Oral Dosage Forms (2026 update).
This guidance applies to manufacturers of solid oral dosage forms (tablets, capsules).
Manufacturers must demonstrate process validation across three stages: process design,
process qualification, and continued process verification. Continuous monitoring data
must be retained for a minimum of 7 years. Manufacturers have 18 months from publication
to update their validation master plans. Effective date: January 1, 2027.
"""

summary = await guidance_summarizer_agent.run(sample_guidance_text)
print(summary)

---
## You've now built

The same 11-subsystem MediOps platform as the LangGraph and ADK versions —
this time expressed through Agent Framework's named orchestration
builders (`HandoffBuilder`, `SequentialBuilder`) for the structured
patterns, and plain Python control flow for the reflection loops, per the
framework's own "use code when code suffices" philosophy.

**Three notebooks, three philosophies, same system:**
- **LangGraph** — everything explicit: you build the graph, the state
  schema, and the routing logic yourself.
- **ADK** — everything implicit: describe agents and let the framework's
  LLM-driven delegation handle routing.
- **Agent Framework** — named builders for the common shapes
  (`Handoff`, `Sequential`), plain Python for everything else.
